In [2]:
import requests
import pandas as pd
import math
import time
from datetime import datetime, timedelta
import os
import pyarrow
print(pyarrow.__version__)

API_KEY = '7450635a5479756e37355678626b4e'
BASE_URL = 'http://openapi.seoul.go.kr:8088'
SERVICE = 'tbCycleRentData'

start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

output_dir = 'input/trip_parquet'
os.makedirs(output_dir, exist_ok=True)

current_date = start_date

while current_date <= end_date:

    date_str = current_date.strftime('%Y-%m-%d')
    output_file = f'{output_dir}/trip_{date_str}.parquet'

    if os.path.exists(output_file):
        print(f'Skipping {date_str} (already exists)')
        current_date += timedelta(days=1)
        continue

    print(f'\n====== Collecting {date_str} ======')
    
    all_rows = []

    for hour in range(24):
        print(f'  Hour {hour}')

        try:
            url = f'{BASE_URL}/{API_KEY}/json/{SERVICE}/1/1/{date_str}/{hour}'
            response = requests.get(url, timeout=30)
            data = response.json()

            if 'rentData' not in data:
                continue
                
            total_count = int(data['rentData']['list_total_count'])
            print(f'    Total records: {total_count}')

            if total_count == 0:
                continue

            batch_size = 1000
            num_batches = math.ceil(total_count / batch_size)

            for i in range(num_batches):
                start = i * batch_size+1
                end = min((i+1) * batch_size, total_count)

                url = f'{BASE_URL}/{API_KEY}/json/{SERVICE}/{start}/{end}/{date_str}/{hour}'
                response = requests.get(url, timeout=30)
                data = response.json()

                rows = data['rentData']['row']
                all_rows.extend(rows)
                
                time.sleep(0.5)
            
        except Exception as e:
            print(f'Error at {date_str} Hour {hour}: {e}')
            time.sleep(5)
            continue
    
    if all_rows:
        df = pd.DataFrame(all_rows)[[
            'RENT_DT',
            'RENT_STATION_ID',
            'RTN_DT',
            'RETURN_STATION_ID',
            'USE_MIN',
            'USE_DST'
        ]]

        df['USE_MIN'] = pd.to_numeric(df['USE_MIN'])
        df['USE_DST'] = pd.to_numeric(df['USE_DST'])
        df['RENT_DT'] = pd.to_datetime(df['RENT_DT'])
        df['RTN_DT'] = pd.to_datetime(df['RTN_DT'])

        df.to_parquet(output_file, index=False)
        print(f'Saved {output_file}, {df.shape}')
    
    current_date += timedelta(days=1)

print('\n 2025 collection completed.')

23.0.1
Skipping 2025-01-01 (already exists)
Skipping 2025-01-02 (already exists)
Skipping 2025-01-03 (already exists)
Skipping 2025-01-04 (already exists)
Skipping 2025-01-05 (already exists)
Skipping 2025-01-06 (already exists)
Skipping 2025-01-07 (already exists)
Skipping 2025-01-08 (already exists)
Skipping 2025-01-09 (already exists)
Skipping 2025-01-10 (already exists)
Skipping 2025-01-11 (already exists)
Skipping 2025-01-12 (already exists)
Skipping 2025-01-13 (already exists)
Skipping 2025-01-14 (already exists)
Skipping 2025-01-15 (already exists)
Skipping 2025-01-16 (already exists)
Skipping 2025-01-17 (already exists)
Skipping 2025-01-18 (already exists)
Skipping 2025-01-19 (already exists)
Skipping 2025-01-20 (already exists)
Skipping 2025-01-21 (already exists)
Skipping 2025-01-22 (already exists)
Skipping 2025-01-23 (already exists)
Skipping 2025-01-24 (already exists)
Skipping 2025-01-25 (already exists)
Skipping 2025-01-26 (already exists)
Skipping 2025-01-27 (already ex